In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time 
from openpyxl import Workbook
from konlpy.tag import Okt
# (konlpy) korean natural language python > 한국어라는 자연어 확인 후 특정 연산처리 결과를 파이썬으로 출력해주는 라이브러리
# (Okt) Open Korean Text => 구체적으로 한국어의 형태소를 분석 
from collections import Counter

options = Options()
options.add_argument("--no-sandbox")
# 구굴 보안 정책 이름 'sandbox' 적용 X
options.add_argument("--disable-dev-shm-usage")
# 구글 브라우저 가상 메모리 한계 5MB -> 가상 메모리 사용 끄기
options.add_argument("--headless")

driver = webdriver.Chrome(options=options)

url = "https://www.kurly.com/categories/912003"
driver.get(url)
time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")
reviews = soup.select("p.css-y49dcn.e36z05c13")
review_lists = [review.get_text().strip().replace("\n", "") for review in reviews]

driver.quit()

wb = Workbook()
ws = wb.active
ws.title = "Kurly Reviews"
ws.append(["리뷰내용"])

for item in review_lists :
    ws.append([item])

okt = Okt()
# join 함수, list 안의 값 str 치환
all_text = " ".join(review_lists)
nouns = okt.nouns(all_text)
# okt 중 메서드 함수 nouns() => 명사 추출
filltered_nouns = [n for n in nouns if len(n)> 1]
count = Counter(filltered_nouns)

ws_keywords = wb.create_sheet("Keyword Frequency")
ws_keywords.append(["키워드", "빈도수"])
for word, freq in count.most_common(30) :
    ws_keywords.append([word, freq])

wb.save("kurly_riviews.xlsx")
print("엑셀 저장 완료 : kurly_riviews.xlsx😊👌")

In [6]:
# -*- coding: utf-8 -*-
"""
스크립트 A의 함수들을 간단히 가져다 쓰는 미니 예시:
카테고리 1개에서 앞의 10개 상품만 수집 → CSV 2종 저장
"""

import pandas as pd
from datetime import datetime

# === 필요한 함수/설정은 A에서 복사해 오세요 ===
# make_driver, collect_product_links_from_category, crawl_one_product, CATEGORY_URL
# (여기서는 import했다고 가정)
# from kurly_full_crawler import make_driver, collect_product_links_from_category, crawl_one_product, CATEGORY_URL

CATEGORY_URL = "https://www.kurly.com/categories/032009?filters=&page=1"

def run_small_example(max_products: int = 10):
    driver = make_driver()
    summaries, all_reviews = [], []
    try:
        links = collect_product_links_from_category(driver, CATEGORY_URL)
        links = links[:max_products]
        print(f"[INFO] 테스트용 링크 {len(links)}개")

        for i, link in enumerate(links, 1):
            print(f"[{i}/{len(links)}] {link}")
            summary, reviews = crawl_one_product(driver, link)
            summaries.append(summary)
            all_reviews.extend(reviews)
    finally:
        driver.quit()

    now = datetime.now().strftime("%Y%m%d_%H%M")
    pd.DataFrame(summaries).to_csv(f"test_health_products_{now}.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_reviews).to_csv(f"test_health_reviews_{now}.csv", index=False, encoding="utf-8-sig")
    print("[DONE] 테스트 CSV 2종 저장 완료")

if __name__ == "__main__":
    run_small_example(10)


[INFO] 테스트용 링크 10개
[1/10] https://www.kurly.com/goods/1001051171
[2/10] https://www.kurly.com/goods/1000963113
[3/10] https://www.kurly.com/goods/5099197
[4/10] https://www.kurly.com/goods/5145668
[5/10] https://www.kurly.com/goods/1000425744
[6/10] https://www.kurly.com/goods/1001325441
[7/10] https://www.kurly.com/goods/1001143403
[8/10] https://www.kurly.com/goods/5055718
[9/10] https://www.kurly.com/goods/5054295
[10/10] https://www.kurly.com/goods/1000127424
[DONE] 테스트 CSV 2종 저장 완료


In [7]:
# -*- coding: utf-8 -*-
"""
Kurly 카테고리(예: 건강식품) 1페이지 → 각 상품 상세 이동
=> 상품명 / 가격 / 브랜드(페이지 내 텍스트) / 리뷰수 + 개별 리뷰(텍스트/날짜/닉네임/별점)
=> 한 개 CSV로 저장

필수 설치:
pip install undetected-chromedriver webdriver-manager beautifulsoup4 lxml pandas
"""

import json
import re
import time
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from bs4 import BeautifulSoup

from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import undetected_chromedriver as uc
from webdriver_manager.chrome import ChromeDriverManager


# ============================ 설정 ============================
CATEGORY_URL = "https://www.kurly.com/categories/032009?filters=&page=1"  # 건강식품 예시
HEADLESS = False
WINDOW_SIZE = "1920,1080"
SHORT, MID, LONG = 3, 7, 12
SCROLL_STEP = 1200

# 테스트: 앞에서 N개만(예: 10). 전체는 None
MAX_PRODUCTS_FROM_PAGE = 10   # ← 전체 돌릴 땐 None

# 리뷰 로딩 제어
MAX_LOAD_MORE_CLICKS = 25
MAX_REVIEWS_PER_PRODUCT = 300

# 정규식
DATE_PAT = re.compile(r"\d{4}\.\d{1,2}\.\d{1,2}")   # 2025.7.09
DIGIT_ONLY = re.compile(r"^[0-9,]+$")


# ======================== 드라이버/유틸 ========================
def make_driver() -> uc.Chrome:
    opts = uc.ChromeOptions()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--lang=ko_KR")
    opts.add_argument(f"--window-size={WINDOW_SIZE}")
    opts.add_argument("--start-maximized")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    driver = uc.Chrome(service=service, options=opts)
    driver.set_page_load_timeout(60)
    return driver

def wait_css(driver, css, timeout=MID):
    return WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.CSS_SELECTOR, css)))

def wait_xpath(driver, xp, timeout=MID):
    return WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.XPATH, xp)))

def safe_click(driver, by, value, timeout=MID) -> bool:
    try:
        el = WebDriverWait(driver, timeout).until(EC.element_to_be_clickable((by, value)))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
        time.sleep(0.2)
        el.click()
        return True
    except Exception:
        return False

def slow_scroll_to_bottom(driver, step_px=SCROLL_STEP, sleep_sec=0.6, max_loops=30):
    last_h = 0
    for _ in range(max_loops):
        driver.execute_script(f"window.scrollBy(0,{step_px});")
        time.sleep(sleep_sec)
        h = driver.execute_script("return document.body.scrollHeight;")
        if h == last_h:
            break
        last_h = h


# =========== 1) 카테고리 1페이지 → 상품 링크 수집 ===========
def collect_product_links_from_category(driver, category_url: str) -> List[str]:
    driver.get(category_url)
    time.sleep(SHORT)
    slow_scroll_to_bottom(driver, max_loops=25)

    anchors = driver.find_elements(By.CSS_SELECTOR, "a[href*='/goods/']")
    links, seen = [], set()
    for a in anchors:
        href = a.get_attribute("href") or ""
        if "/goods/" in href and href not in seen:
            seen.add(href)
            links.append(href)
    return links


# ==================== 2) JSON-LD(보조 정보) ====================
def parse_json_ld_for_price_reviewcount(html: str) -> Dict[str, Any]:
    """price / reviewCount만 보조적으로 읽는다(브랜드는 DOM 텍스트로만)."""
    soup = BeautifulSoup(html, "lxml")
    data: Dict[str, Any] = {}
    for tag in soup.select('script[type="application/ld+json"]'):
        try:
            j = json.loads(tag.string)
        except Exception:
            continue
        objs = j if isinstance(j, list) else [j]
        for obj in objs:
            if not isinstance(obj, dict):
                continue
            if obj.get("@type") == "Product":
                # price
                offers = obj.get("offers")
                price = None
                if isinstance(offers, dict):
                    price = offers.get("price")
                elif isinstance(offers, list):
                    for off in offers:
                        if isinstance(off, dict) and off.get("price"):
                            price = off["price"]; break
                if price:
                    try:
                        data["price"] = int(str(price).replace(",", "").strip())
                    except Exception:
                        pass
                # reviewCount
                ar = obj.get("aggregateRating")
                if isinstance(ar, dict) and ar.get("reviewCount") is not None:
                    try:
                        data["review_count"] = int(ar["reviewCount"])
                    except Exception:
                        pass
                return data
    return data


# ================ 3) DOM 기반 추출(브랜드/가격/리뷰수) ================
def extract_brand_text(driver) -> Optional[str]:
    """
    ★ 링크 '이동' 없이 페이지에서 보이는 브랜드 텍스트만 추출.
    - 최우선: 타이틀(h1) 상위 섹션 내부의 /brand/ 앵커 텍스트
    - 보조: 페이지 어디든 /brand/ 앵커 텍스트
    - 최종 실패 시 None
    """
    # 1) 타이틀 상위 섹션에서 탐색
    try:
        title = wait_xpath(driver, "//h1", timeout=MID)
        info = title.find_element(By.XPATH, "./ancestor::section[1] | ./ancestor::div[2]")
        anchors = info.find_elements(By.XPATH, ".//a[contains(@href,'/brand/') and normalize-space()]")
        for a in anchors:
            t = a.text.strip().replace(">", "").strip()
            if t and "브랜드" not in t and "브랜드관" not in t:
                return t
    except Exception:
        pass
    # 2) 전체 페이지에서 탐색
    try:
        anchors = driver.find_elements(By.XPATH, "//a[contains(@href,'/brand/') and normalize-space()]")
        for a in anchors:
            t = a.text.strip().replace(">", "").strip()
            if t and "브랜드" not in t and "브랜드관" not in t:
                return t
    except Exception:
        pass
    return None

def extract_price_from_dom(driver) -> Optional[int]:
    """타이틀 상위 섹션에서 숫자만 있는 <span>들을 훑어 가장 앞의 값을 현재가로 간주."""
    try:
        title = wait_xpath(driver, "//h1", timeout=LONG)
        box = title.find_element(By.XPATH, "./ancestor::section[1] | ./ancestor::div[2]")
        spans = box.find_elements(By.XPATH, ".//span[normalize-space()]")
        candidates = []
        for s in spans:
            txt = s.text.strip()
            if not txt or "%" in txt:
                continue
            if DIGIT_ONLY.match(txt):
                candidates.append(int(txt.replace(",", "")))
        if candidates:
            return candidates[0]
    except Exception:
        pass
    return None

def extract_reviewcount_from_dom(driver) -> Optional[int]:
    """탭/요약에서 '후기 (숫자)' 패턴으로 리뷰 수 추출."""
    try:
        nodes = driver.find_elements(By.XPATH, "//*[contains(.,'후기') and (self::a or self::span or self::div)]")
        for n in nodes:
            txt = (n.text or "").replace(",", "").strip()
            m = re.search(r"후기[\s(]*(\d+)", txt)
            if m:
                return int(m.group(1))
    except Exception:
        pass
    return None


# ================ 4) 후기 탭 열기/확장 + 리뷰 파싱 ================
def open_review_tab(driver) -> None:
    for xp in [
        "//nav//li[.//span[contains(.,'후기')]]",
        "//nav//*[self::a or self::button][contains(.,'후기')]",
        "//button[contains(.,'후기')]",
        "//a[contains(.,'후기')]",
    ]:
        if safe_click(driver, By.XPATH, xp, timeout=SHORT):
            time.sleep(1.0)
            break
    try:
        WebDriverWait(driver, LONG).until(
            EC.presence_of_element_located((By.XPATH, "//*[contains(@aria-label,'상품 후기') or contains(.,'상품 후기')][self::h2 or self::h3 or self::div]"))
        )
    except Exception:
        pass

def expand_reviews(driver, max_clicks=MAX_LOAD_MORE_CLICKS, max_reviews=MAX_REVIEWS_PER_PRODUCT):
    clicks = 0
    prev_count = 0
    while clicks < max_clicks:
        clicked = False
        # 리스트 더보기
        for b in driver.find_elements(By.XPATH, "//button[contains(.,'더보기') or contains(.,'후기 더보기')]"):
            try:
                if b.is_displayed() and b.is_enabled():
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    time.sleep(0.2)
                    b.click()
                    clicked = True
                    clicks += 1
                    time.sleep(1.2)
            except Exception:
                continue
        # 카드 내부 본문 더보기
        for b in driver.find_elements(By.XPATH, "//article//button[contains(.,'더보기') or contains(.,'자세히')]"):
            try:
                if b.is_displayed():
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    time.sleep(0.15)
                    b.click()
                    time.sleep(0.2)
            except Exception:
                continue
        slow_scroll_to_bottom(driver, max_loops=2)

        # 중단 조건
        soup = BeautifulSoup(driver.page_source, "lxml")
        cur = len(soup.select("article")) + len(soup.select("li"))
        if max_reviews and cur >= max_reviews:
            break
        if not clicked:
            if cur == prev_count:
                break
            prev_count = cur

def parse_reviews_from_page(driver, product_url: str, product_title: str) -> List[Dict[str, Any]]:
    soup = BeautifulSoup(driver.page_source, "lxml")
    root = None
    for sel in ["section[id*='review']", "section[class*='review']", "div[id*='review']", "div[class*='review']"]:
        r = soup.select_one(sel)
        if r: root = r; break
    if root is None:
        root = soup

    items = root.select("article")
    if not items:
        items = root.select("li")

    rows: List[Dict[str, Any]] = []
    for tag in items:
        # 본문: 가장 긴 <p>
        ps = [p.get_text(" ", strip=True) for p in tag.select("p") if p.get_text(strip=True)]
        body = max(ps, key=len) if ps else tag.get_text(" ", strip=True)
        if not body or len(body) < 10:
            continue

        # 날짜
        m = DATE_PAT.search(tag.get_text(" ", strip=True))
        date_str = m.group(0) if m else None

        # 닉네임(마스킹 포함)
        nickname = None
        for txt in tag.stripped_strings:
            s = str(txt).strip()
            if 2 <= len(s) <= 8 and (s.endswith("님") or "**" in s):
                nickname = s; break

        # 리뷰 별점(있으면)
        rating = None
        metas = tag.select("[aria-label],[alt],[title]") + tag.find_all(string=True)
        for s in metas:
            s_ = s if isinstance(s, str) else (s.get("aria-label") or s.get("alt") or s.get("title") or "")
            mm = re.search(r"(\d+)\s*점|별\s*(\d+)", str(s_))
            if mm:
                rating = int(mm.group(1) or mm.group(2)); break

        rows.append({
            "product_title": product_title,
            "product_url": product_url,
            "review_date": date_str,
            "review_author": nickname,
            "review_rating": rating,
            "review_text": body
        })
        if MAX_REVIEWS_PER_PRODUCT and len(rows) >= MAX_REVIEWS_PER_PRODUCT:
            break
    return rows


# ================ 5) 상품 단위 크롤링(요약 + 리뷰) ================
def crawl_one_product(driver, url: str) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    driver.get(url)
    time.sleep(SHORT)

    # 제목
    try:
        title = wait_xpath(driver, "//h1", timeout=LONG).text.strip()
    except Exception:
        try:
            title = wait_css(driver, "h1, h2", timeout=MID).text.strip()
        except Exception:
            title = ""

    # 브랜드(페이지 내 텍스트만)
    brand = extract_brand_text(driver)

    # 가격: JSON-LD → DOM 후보
    j = parse_json_ld_for_price_reviewcount(driver.page_source)
    price = j.get("price")
    if price is None:
        price = extract_price_from_dom(driver)

    # 리뷰 수: JSON-LD → DOM
    review_count = j.get("review_count")
    if review_count is None:
        review_count = extract_reviewcount_from_dom(driver)

    # 후기 탭 열고 확장 → 리뷰 파싱
    open_review_tab(driver)
    expand_reviews(driver)
    reviews = parse_reviews_from_page(driver, url, title)

    summary = {
        "product_title": title,
        "product_url": url,
        "brand": brand,
        "price": price,
        "review_count": review_count,
        "scraped_reviews": len(reviews),
    }
    return summary, reviews


# ============================ 메인 ============================
def main():
    driver = make_driver()
    rows_for_csv: List[Dict[str, Any]] = []
    try:
        links = collect_product_links_from_category(driver, CATEGORY_URL)
        if isinstance(MAX_PRODUCTS_FROM_PAGE, int):
            links = links[:MAX_PRODUCTS_FROM_PAGE]
        print(f"[INFO] 처리할 상품 링크 {len(links)}개")

        for i, link in enumerate(links, 1):
            print(f"\n[{i}/{len(links)}] {link}")
            summary, reviews = crawl_one_product(driver, link)
            print("   ->", summary)

            # 리뷰가 있으면 리뷰별로 행 생성, 없으면 요약만 한 줄 생성
            if reviews:
                for r in reviews:
                    rows_for_csv.append({
                        "category_url": CATEGORY_URL,
                        "product_title": summary["product_title"],
                        "product_url": summary["product_url"],
                        "brand": summary["brand"],
                        "price": summary["price"],
                        "review_count": summary["review_count"],
                        "review_date": r.get("review_date"),
                        "review_author": r.get("review_author"),
                        "review_rating": r.get("review_rating"),
                        "review_text": r.get("review_text"),
                    })
            else:
                rows_for_csv.append({
                    "category_url": CATEGORY_URL,
                    "product_title": summary["product_title"],
                    "product_url": summary["product_url"],
                    "brand": summary["brand"],
                    "price": summary["price"],
                    "review_count": summary["review_count"],
                    "review_date": None,
                    "review_author": None,
                    "review_rating": None,
                    "review_text": None,
                })
    finally:
        driver.quit()

    now = datetime.now().strftime("%Y%m%d_%H%M")
    out = f"kurly_health_merged_{now}.csv"
    pd.DataFrame(rows_for_csv, columns=[
        "category_url","product_title","product_url","brand","price","review_count",
        "review_date","review_author","review_rating","review_text"
    ]).to_csv(out, index=False, encoding="utf-8-sig")
    print(f"\n[DONE] 통합 CSV 저장 → {out} (행 수: {len(rows_for_csv)})")


if __name__ == "__main__":
    main()


[INFO] 처리할 상품 링크 10개

[1/10] https://www.kurly.com/goods/1001051171
   -> {'product_title': '[덴프스] 트루바이타민 I (30일분)', 'product_url': 'https://www.kurly.com/goods/1001051171', 'brand': None, 'price': 25400, 'review_count': 136, 'scraped_reviews': 10}

[2/10] https://www.kurly.com/goods/1000963113
   -> {'product_title': '[비타바움] 멀티비타민키즈 + V - Complex15 (30일분)', 'product_url': 'https://www.kurly.com/goods/1000963113', 'brand': None, 'price': 95200, 'review_count': 205, 'scraped_reviews': 10}

[3/10] https://www.kurly.com/goods/5099197
   -> {'product_title': '[로트벡쉔] 이뮨샷 올인원 비타민 플러스 (10일분)', 'product_url': 'https://www.kurly.com/goods/5099197', 'brand': None, 'price': 59000, 'review_count': 995, 'scraped_reviews': 10}

[4/10] https://www.kurly.com/goods/5145668
   -> {'product_title': '[로트벡쉔] 이뮨킥 10 멀티비타민+아연 (10일분)', 'product_url': 'https://www.kurly.com/goods/5145668', 'brand': None, 'price': 49000, 'review_count': 447, 'scraped_reviews': 10}

[5/10] https://www.kurly.com/goods/1000425744


In [8]:
# -*- coding: utf-8 -*-
"""
Kurly 카테고리(예: 건강식품) 1페이지 → 각 상품 상세 이동
=> product_title(원문) / product_name([브랜드] 제거) / brand(대괄호에서 추출) / price / review_count
   + 개별 리뷰(review_date/review_author/review_rating/review_text)
=> 한 개 CSV로 저장

필수 설치:
pip install undetected-chromedriver webdriver-manager beautifulsoup4 lxml pandas
"""

import json
import re
import time
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from bs4 import BeautifulSoup

from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import undetected_chromedriver as uc
from webdriver_manager.chrome import ChromeDriverManager


# ============================ 설정 ============================
CATEGORY_URL = "https://www.kurly.com/categories/032009?filters=&page=1"  # 건강식품 예시
HEADLESS = False
WINDOW_SIZE = "1920,1080"
SHORT, MID, LONG = 3, 7, 12
SCROLL_STEP = 1200

# 테스트: 앞에서 N개만(예: 10). 전체는 None
MAX_PRODUCTS_FROM_PAGE = 10   # 전체 수집하려면 None

# 리뷰 로딩 제어
MAX_LOAD_MORE_CLICKS = 25
MAX_REVIEWS_PER_PRODUCT = 300

# 정규식
DATE_PAT = re.compile(r"\d{4}\.\d{1,2}\.\d{1,2}")   # 2025.7.09
DIGIT_ONLY = re.compile(r"^[0-9,]+$")


# ======================== 드라이버/유틸 ========================
def make_driver() -> uc.Chrome:
    opts = uc.ChromeOptions()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--lang=ko_KR")
    opts.add_argument(f"--window-size={WINDOW_SIZE}")
    opts.add_argument("--start-maximized")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    driver = uc.Chrome(service=service, options=opts)
    driver.set_page_load_timeout(60)
    return driver

def wait_css(driver, css, timeout=MID):
    return WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.CSS_SELECTOR, css)))

def wait_xpath(driver, xp, timeout=MID):
    return WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.XPATH, xp)))

def safe_click(driver, by, value, timeout=MID) -> bool:
    try:
        el = WebDriverWait(driver, timeout).until(EC.element_to_be_clickable((by, value)))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
        time.sleep(0.2)
        el.click()
        return True
    except Exception:
        return False

def slow_scroll_to_bottom(driver, step_px=SCROLL_STEP, sleep_sec=0.6, max_loops=30):
    last_h = 0
    for _ in range(max_loops):
        driver.execute_script(f"window.scrollBy(0,{step_px});")
        time.sleep(sleep_sec)
        h = driver.execute_script("return document.body.scrollHeight;")
        if h == last_h:
            break
        last_h = h


# =========== 1) 카테고리 1페이지 → 상품 링크 수집 ===========
def collect_product_links_from_category(driver, category_url: str) -> List[str]:
    driver.get(category_url)
    time.sleep(SHORT)
    slow_scroll_to_bottom(driver, max_loops=25)

    anchors = driver.find_elements(By.CSS_SELECTOR, "a[href*='/goods/']")
    links, seen = [], set()
    for a in anchors:
        href = a.get_attribute("href") or ""
        if "/goods/" in href and href not in seen:
            seen.add(href)
            links.append(href)
    return links


# ==================== 2) JSON-LD(보조: 가격/리뷰수) ====================
def parse_json_ld_for_price_reviewcount(html: str) -> Dict[str, Any]:
    """price / reviewCount만 보조적으로 읽는다."""
    soup = BeautifulSoup(html, "lxml")
    data: Dict[str, Any] = {}
    for tag in soup.select('script[type="application/ld+json"]'):
        try:
            j = json.loads(tag.string)
        except Exception:
            continue
        objs = j if isinstance(j, list) else [j]
        for obj in objs:
            if not isinstance(obj, dict):
                continue
            if obj.get("@type") == "Product":
                # price
                offers = obj.get("offers")
                price = None
                if isinstance(offers, dict):
                    price = offers.get("price")
                elif isinstance(offers, list):
                    for off in offers:
                        if isinstance(off, dict) and off.get("price"):
                            price = off["price"]; break
                if price:
                    try:
                        data["price"] = int(str(price).replace(",", "").strip())
                    except Exception:
                        pass
                # reviewCount
                ar = obj.get("aggregateRating")
                if isinstance(ar, dict) and ar.get("reviewCount") is not None:
                    try:
                        data["review_count"] = int(ar["reviewCount"])
                    except Exception:
                        pass
                return data
    return data


# ================ 3) DOM 기반 추출(가격/리뷰수) ================
def extract_price_from_dom(driver) -> Optional[int]:
    """타이틀 상위 섹션에서 숫자만 있는 <span>들을 훑어 가장 앞의 값을 현재가로 간주."""
    try:
        title = wait_xpath(driver, "//h1", timeout=LONG)
        box = title.find_element(By.XPATH, "./ancestor::section[1] | ./ancestor::div[2]")
        spans = box.find_elements(By.XPATH, ".//span[normalize-space()]")
        candidates = []
        for s in spans:
            txt = s.text.strip()
            if not txt or "%" in txt:
                continue
            if DIGIT_ONLY.match(txt):
                candidates.append(int(txt.replace(",", "")))
        if candidates:
            return candidates[0]
    except Exception:
        pass
    return None

def extract_reviewcount_from_dom(driver) -> Optional[int]:
    """탭/요약에서 '후기 (숫자)' 패턴으로 리뷰 수 추출."""
    try:
        nodes = driver.find_elements(By.XPATH, "//*[contains(.,'후기') and (self::a or self::span or self::div)]")
        for n in nodes:
            txt = (n.text or "").replace(",", "").strip()
            m = re.search(r"후기[\s(]*(\d+)", txt)
            if m:
                return int(m.group(1))
    except Exception:
        pass
    return None


# ================ 4) 후기 탭 열기/확장 + 리뷰 파싱 ================
def open_review_tab(driver) -> None:
    for xp in [
        "//nav//li[.//span[contains(.,'후기')]]",
        "//nav//*[self::a or self::button][contains(.,'후기')]",
        "//button[contains(.,'후기')]",
        "//a[contains(.,'후기')]",
    ]:
        if safe_click(driver, By.XPATH, xp, timeout=SHORT):
            time.sleep(1.0)
            break
    try:
        WebDriverWait(driver, LONG).until(
            EC.presence_of_element_located((By.XPATH, "//*[contains(@aria-label,'상품 후기') or contains(.,'상품 후기')][self::h2 or self::h3 or self::div]"))
        )
    except Exception:
        pass

def expand_reviews(driver, max_clicks=MAX_LOAD_MORE_CLICKS, max_reviews=MAX_REVIEWS_PER_PRODUCT):
    clicks = 0
    prev_count = 0
    while clicks < max_clicks:
        clicked = False
        # 리스트 더보기
        for b in driver.find_elements(By.XPATH, "//button[contains(.,'더보기') or contains(.,'후기 더보기')]"):
            try:
                if b.is_displayed() and b.is_enabled():
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    time.sleep(0.2)
                    b.click()
                    clicked = True
                    clicks += 1
                    time.sleep(1.2)
            except Exception:
                continue
        # 카드 내부 본문 더보기
        for b in driver.find_elements(By.XPATH, "//article//button[contains(.,'더보기') or contains(.,'자세히')]"):
            try:
                if b.is_displayed():
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    time.sleep(0.15)
                    b.click()
                    time.sleep(0.2)
            except Exception:
                continue
        slow_scroll_to_bottom(driver, max_loops=2)

        # 중단 조건
        soup = BeautifulSoup(driver.page_source, "lxml")
        cur = len(soup.select("article")) + len(soup.select("li"))
        if max_reviews and cur >= max_reviews:
            break
        if not clicked:
            if cur == prev_count:
                break
            prev_count = cur

def parse_reviews_from_page(driver, product_url: str, product_title: str) -> List[Dict[str, Any]]:
    soup = BeautifulSoup(driver.page_source, "lxml")
    root = None
    for sel in ["section[id*='review']", "section[class*='review']", "div[id*='review']", "div[class*='review']"]:
        r = soup.select_one(sel)
        if r: root = r; break
    if root is None:
        root = soup

    items = root.select("article")
    if not items:
        items = root.select("li")

    rows: List[Dict[str, Any]] = []
    for tag in items:
        # 본문: 가장 긴 <p>
        ps = [p.get_text(" ", strip=True) for p in tag.select("p") if p.get_text(strip=True)]
        body = max(ps, key=len) if ps else tag.get_text(" ", strip=True)
        if not body or len(body) < 10:
            continue

        # 날짜
        m = DATE_PAT.search(tag.get_text(" ", strip=True))
        date_str = m.group(0) if m else None

        # 닉네임(마스킹 포함)
        nickname = None
        for txt in tag.stripped_strings:
            s = str(txt).strip()
            if 2 <= len(s) <= 8 and (s.endswith("님") or "**" in s):
                nickname = s; break

        # 리뷰 별점(있으면)
        rating = None
        metas = tag.select("[aria-label],[alt],[title]") + tag.find_all(string=True)
        for s in metas:
            s_ = s if isinstance(s, str) else (s.get("aria-label") or s.get("alt") or s.get("title") or "")
            mm = re.search(r"(\d+)\s*점|별\s*(\d+)", str(s_))
            if mm:
                rating = int(mm.group(1) or mm.group(2)); break

        rows.append({
            "product_url": product_url,
            "product_title": product_title,
            "review_date": date_str,
            "review_author": nickname,
            "review_rating": rating,
            "review_text": body
        })
        if MAX_REVIEWS_PER_PRODUCT and len(rows) >= MAX_REVIEWS_PER_PRODUCT:
            break
    return rows


# ================ 5) 타이틀에서 브랜드/제품명 분리 ================
def split_brand_from_title(title: str) -> Tuple[Optional[str], str]:
    """
    예: "[덴프스] 트루바이타민 I (30일분)"
    -> brand="덴프스", product_name="트루바이타민 I (30일분)"
    - 맨 앞 [] 한 쌍만 브랜드로 사용
    - 없으면 brand=None, product_name=원문
    """
    if not title:
        return None, ""
    m = re.match(r"\s*\[([^\]]+)\]\s*(.*)", title)
    if m:
        brand = m.group(1).strip()
        name = m.group(2).strip()
        return brand, name if name else title.strip()
    return None, title.strip()


# ================ 6) 상품 단위 크롤링(요약 + 리뷰) ================
def crawl_one_product(driver, url: str) -> List[Dict[str, Any]]:
    driver.get(url)
    time.sleep(SHORT)

    # 제목
    try:
        title = wait_xpath(driver, "//h1", timeout=LONG).text.strip()
    except Exception:
        try:
            title = wait_css(driver, "h1, h2", timeout=MID).text.strip()
        except Exception:
            title = ""

    # 타이틀에서 브랜드/제품명 분리
    brand, product_name = split_brand_from_title(title)

    # 가격/리뷰수: JSON-LD → DOM
    j = parse_json_ld_for_price_reviewcount(driver.page_source)
    price = j.get("price")
    if price is None:
        price = extract_price_from_dom(driver)

    review_count = j.get("review_count")
    if review_count is None:
        review_count = extract_reviewcount_from_dom(driver)

    # 후기 탭 열고 확장 → 리뷰 파싱
    open_review_tab(driver)
    expand_reviews(driver)
    reviews = parse_reviews_from_page(driver, url, title)

    # 리뷰가 있으면 리뷰별로 행 생성, 없으면 요약만 한 줄 생성
    rows: List[Dict[str, Any]] = []
    if reviews:
        for r in reviews:
            rows.append({
                "category_url": CATEGORY_URL,
                "product_title": title,          # 원문(브랜드 대괄호 포함)
                "product_name": product_name,    # 대괄호 제거한 제품명
                "brand": brand,                  # 대괄호 안 텍스트
                "product_url": url,
                "price": price,
                "review_count": review_count,
                "review_date": r.get("review_date"),
                "review_author": r.get("review_author"),
                "review_rating": r.get("review_rating"),
                "review_text": r.get("review_text"),
            })
    else:
        rows.append({
            "category_url": CATEGORY_URL,
            "product_title": title,
            "product_name": product_name,
            "brand": brand,
            "product_url": url,
            "price": price,
            "review_count": review_count,
            "review_date": None,
            "review_author": None,
            "review_rating": None,
            "review_text": None,
        })
    return rows


# ============================ 메인 ============================
def main():
    driver = make_driver()
    rows_for_csv: List[Dict[str, Any]] = []
    try:
        links = collect_product_links_from_category(driver, CATEGORY_URL)
        if isinstance(MAX_PRODUCTS_FROM_PAGE, int):
            links = links[:MAX_PRODUCTS_FROM_PAGE]
        print(f"[INFO] 처리할 상품 링크 {len(links)}개")

        for i, link in enumerate(links, 1):
            print(f"\n[{i}/{len(links)}] {link}")
            rows = crawl_one_product(driver, link)
            rows_for_csv.extend(rows)
    finally:
        driver.quit()

    now = datetime.now().strftime("%Y%m%d_%H%M")
    out = f"kurly_health_merged_{now}.csv"
    cols = [
        "category_url","product_title","product_name","brand","product_url",
        "price","review_count","review_date","review_author","review_rating","review_text"
    ]
    pd.DataFrame(rows_for_csv, columns=cols).to_csv(out, index=False, encoding="utf-8-sig")
    print(f"\n[DONE] 통합 CSV 저장 → {out} (행 수: {len(rows_for_csv)})")


if __name__ == "__main__":
    main()


[INFO] 처리할 상품 링크 10개

[1/10] https://www.kurly.com/goods/1001051171

[2/10] https://www.kurly.com/goods/1000963113

[3/10] https://www.kurly.com/goods/5099197

[4/10] https://www.kurly.com/goods/5145668

[5/10] https://www.kurly.com/goods/1000425744

[6/10] https://www.kurly.com/goods/1001325441

[7/10] https://www.kurly.com/goods/1001143403

[8/10] https://www.kurly.com/goods/5055718

[9/10] https://www.kurly.com/goods/5054295

[10/10] https://www.kurly.com/goods/1000127424

[DONE] 통합 CSV 저장 → kurly_health_merged_20250922_1743.csv (행 수: 99)


In [11]:
# -*- coding: utf-8 -*-
"""
Kurly 카테고리(예: 건강식품) 1페이지 → 각 상품 상세 이동
=> product_name([브랜드] 제거), brand(대괄호 텍스트), price, review_count, review_date, review_text
=> 한 개 CSV로 저장

설치:
pip install undetected-chromedriver webdriver-manager beautifulsoup4 lxml pandas
"""

import json
import re
import time
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from bs4 import BeautifulSoup

from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import undetected_chromedriver as uc
from webdriver_manager.chrome import ChromeDriverManager


# ============================ 설정 ============================
CATEGORY_URL = "https://www.kurly.com/categories/032009?filters=&page=1"  # 건강식품 예시
HEADLESS = False
WINDOW_SIZE = "1920,1080"
SHORT, MID, LONG = 3, 7, 12
SCROLL_STEP = 1200

# 테스트: 앞에서 N개만(예: 10). 전체는 None
MAX_PRODUCTS_FROM_PAGE = None   # 전체 수집하려면 None

# 리뷰 로딩 제어
MAX_LOAD_MORE_CLICKS = 25
MAX_REVIEWS_PER_PRODUCT = 300

# 정규식
DATE_PAT = re.compile(r"\d{4}\.\d{1,2}\.\d{1,2}")   # 2025.7.09
DIGIT_ONLY = re.compile(r"^[0-9,]+$")


# ======================== 드라이버/유틸 ========================
def make_driver() -> uc.Chrome:
    opts = uc.ChromeOptions()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--lang=ko_KR")
    opts.add_argument(f"--window-size={WINDOW_SIZE}")
    opts.add_argument("--start-maximized")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    driver = uc.Chrome(service=service, options=opts)
    driver.set_page_load_timeout(60)
    return driver

def wait_css(driver, css, timeout=MID):
    return WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.CSS_SELECTOR, css)))

def wait_xpath(driver, xp, timeout=MID):
    return WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.XPATH, xp)))

def safe_click(driver, by, value, timeout=MID) -> bool:
    try:
        el = WebDriverWait(driver, timeout).until(EC.element_to_be_clickable((by, value)))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
        time.sleep(0.2)
        el.click()
        return True
    except Exception:
        return False

def slow_scroll_to_bottom(driver, step_px=SCROLL_STEP, sleep_sec=0.6, max_loops=30):
    last_h = 0
    for _ in range(max_loops):
        driver.execute_script(f"window.scrollBy(0,{step_px});")
        time.sleep(sleep_sec)
        h = driver.execute_script("return document.body.scrollHeight;")
        if h == last_h:
            break
        last_h = h


# =========== 1) 카테고리 1페이지 → 상품 링크 수집 ===========
def collect_product_links_from_category(driver, category_url: str) -> List[str]:
    driver.get(category_url)
    time.sleep(SHORT)
    slow_scroll_to_bottom(driver, max_loops=25)

    anchors = driver.find_elements(By.CSS_SELECTOR, "a[href*='/goods/']")
    links, seen = [], set()
    for a in anchors:
        href = a.get_attribute("href") or ""
        if "/goods/" in href and href not in seen:
            seen.add(href)
            links.append(href)
    return links


# ==================== 2) JSON-LD(보조: 가격/리뷰수) ====================
def parse_json_ld_for_price_reviewcount(html: str) -> Dict[str, Any]:
    """price / reviewCount만 보조적으로 읽는다."""
    soup = BeautifulSoup(html, "lxml")
    data: Dict[str, Any] = {}
    for tag in soup.select('script[type="application/ld+json"]'):
        try:
            j = json.loads(tag.string)
        except Exception:
            continue
        objs = j if isinstance(j, list) else [j]
        for obj in objs:
            if not isinstance(obj, dict):
                continue
            if obj.get("@type") == "Product":
                # price
                offers = obj.get("offers")
                price = None
                if isinstance(offers, dict):
                    price = offers.get("price")
                elif isinstance(offers, list):
                    for off in offers:
                        if isinstance(off, dict) and off.get("price"):
                            price = off["price"]; break
                if price:
                    try:
                        data["price"] = int(str(price).replace(",", "").strip())
                    except Exception:
                        pass
                # reviewCount
                ar = obj.get("aggregateRating")
                if isinstance(ar, dict) and ar.get("reviewCount") is not None:
                    try:
                        data["review_count"] = int(ar["reviewCount"])
                    except Exception:
                        pass
                return data
    return data


# ================ 3) DOM 기반 추출(가격/리뷰수) ================
def extract_price_from_dom(driver) -> Optional[int]:
    """타이틀 상위 섹션에서 숫자만 있는 <span>들을 훑어 가장 앞의 값을 현재가로 간주."""
    try:
        title = wait_xpath(driver, "//h1", timeout=LONG)
        box = title.find_element(By.XPATH, "./ancestor::section[1] | ./ancestor::div[2]")
        spans = box.find_elements(By.XPATH, ".//span[normalize-space()]")
        candidates = []
        for s in spans:
            txt = s.text.strip()
            if not txt or "%" in txt:
                continue
            if DIGIT_ONLY.match(txt):
                candidates.append(int(txt.replace(",", "")))
        if candidates:
            return candidates[0]
    except Exception:
        pass
    return None

def extract_reviewcount_from_dom(driver) -> Optional[int]:
    """탭/요약에서 '후기 (숫자)' 패턴으로 리뷰 수 추출."""
    try:
        nodes = driver.find_elements(By.XPATH, "//*[contains(.,'후기') and (self::a or self::span or self::div)]")
        for n in nodes:
            txt = (n.text or "").replace(",", "").strip()
            m = re.search(r"후기[\s(]*(\d+)", txt)
            if m:
                return int(m.group(1))
    except Exception:
        pass
    return None


# ================ 4) 후기 탭 열기/확장 + 리뷰 파싱 ================
def open_review_tab(driver) -> None:
    for xp in [
        "//nav//li[.//span[contains(.,'후기')]]",
        "//nav//*[self::a or self::button][contains(.,'후기')]",
        "//button[contains(.,'후기')]",
        "//a[contains(.,'후기')]",
    ]:
        if safe_click(driver, By.XPATH, xp, timeout=SHORT):
            time.sleep(1.0)
            break
    try:
        WebDriverWait(driver, LONG).until(
            EC.presence_of_element_located((By.XPATH, "//*[contains(@aria-label,'상품 후기') or contains(.,'상품 후기')][self::h2 or self::h3 or self::div]"))
        )
    except Exception:
        pass

def expand_reviews(driver, max_clicks=MAX_LOAD_MORE_CLICKS, max_reviews=MAX_REVIEWS_PER_PRODUCT):
    clicks = 0
    prev_count = 0
    while clicks < max_clicks:
        clicked = False
        # 리스트 더보기
        for b in driver.find_elements(By.XPATH, "//button[contains(.,'더보기') or contains(.,'후기 더보기')]"):
            try:
                if b.is_displayed() and b.is_enabled():
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    time.sleep(0.2)
                    b.click()
                    clicked = True
                    clicks += 1
                    time.sleep(1.2)
            except Exception:
                continue
        # 카드 내부 본문 더보기
        for b in driver.find_elements(By.XPATH, "//article//button[contains(.,'더보기') or contains(.,'자세히')]"):
            try:
                if b.is_displayed():
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    time.sleep(0.15)
                    b.click()
                    time.sleep(0.2)
            except Exception:
                continue
        slow_scroll_to_bottom(driver, max_loops=2)

        # 중단 조건
        soup = BeautifulSoup(driver.page_source, "lxml")
        cur = len(soup.select("article")) + len(soup.select("li"))
        if max_reviews and cur >= max_reviews:
            break
        if not clicked:
            if cur == prev_count:
                break
            prev_count = cur

def parse_reviews_from_page(driver, product_url: str) -> List[Dict[str, Any]]:
    soup = BeautifulSoup(driver.page_source, "lxml")
    root = None
    for sel in ["section[id*='review']", "section[class*='review']", "div[id*='review']", "div[class*='review']"]:
        r = soup.select_one(sel)
        if r: root = r; break
    if root is None:
        root = soup

    items = root.select("article")
    if not items:
        items = root.select("li")

    rows: List[Dict[str, Any]] = []
    for tag in items:
        # 본문: 가장 긴 <p>
        ps = [p.get_text(" ", strip=True) for p in tag.select("p") if p.get_text(strip=True)]
        body = max(ps, key=len) if ps else tag.get_text(" ", strip=True)
        if not body or len(body) < 10:
            continue

        # 날짜만 추출
        m = DATE_PAT.search(tag.get_text(" ", strip=True))
        date_str = m.group(0) if m else None

        rows.append({
            "product_url": product_url,
            "review_date": date_str,
            "review_text": body
        })

        if MAX_REVIEWS_PER_PRODUCT and len(rows) >= MAX_REVIEWS_PER_PRODUCT:
            break
    return rows


# ================ 5) 타이틀에서 브랜드/제품명 분리 ================
def split_brand_from_title(title: str) -> Tuple[Optional[str], str]:
    """
    예: "[덴프스] 트루바이타민 I (30일분)"
    -> brand="덴프스", product_name="트루바이타민 I (30일분)"
    """
    if not title:
        return None, ""
    m = re.match(r"\s*\[([^\]]+)\]\s*(.*)", title)
    if m:
        brand = m.group(1).strip()
        name = m.group(2).strip()
        return brand, name if name else title.strip()
    return None, title.strip()


# ================ 6) 상품 단위 크롤링(요약 + 리뷰) ================
def crawl_one_product(driver, url: str) -> List[Dict[str, Any]]:
    driver.get(url)
    time.sleep(SHORT)

    # 제목 → product_name, brand
    try:
        title = wait_xpath(driver, "//h1", timeout=LONG).text.strip()
    except Exception:
        try:
            title = wait_css(driver, "h1, h2", timeout=MID).text.strip()
        except Exception:
            title = ""
    brand, product_name = split_brand_from_title(title)

    # 가격/리뷰수: JSON-LD → DOM
    j = parse_json_ld_for_price_reviewcount(driver.page_source)
    price = j.get("price")
    if price is None:
        price = extract_price_from_dom(driver)

    review_count = j.get("review_count")
    if review_count is None:
        review_count = extract_reviewcount_from_dom(driver)

    # 후기 탭 열고 확장 → 리뷰(날짜/본문만)
    open_review_tab(driver)
    expand_reviews(driver)
    reviews = parse_reviews_from_page(driver, url)

    # 리뷰가 있으면 리뷰별 행, 없으면 요약 한 줄
    rows: List[Dict[str, Any]] = []
    if reviews:
        for r in reviews:
            rows.append({
                "category_url": CATEGORY_URL,
                "product_name": product_name,
                "brand": brand,
                "product_url": url,
                "price": price,
                "review_count": review_count,
                "review_date": r.get("review_date"),
                "review_text": r.get("review_text"),
            })
    else:
        rows.append({
            "category_url": CATEGORY_URL,
            "product_name": product_name,
            "brand": brand,
            "product_url": url,
            "price": price,
            "review_count": review_count,
            "review_date": None,
            "review_text": None,
        })
    return rows


# ============================ 메인 ============================
def main():
    driver = make_driver()
    rows_for_csv: List[Dict[str, Any]] = []
    try:
        links = collect_product_links_from_category(driver, CATEGORY_URL)
        if isinstance(MAX_PRODUCTS_FROM_PAGE, int):
            links = links[:MAX_PRODUCTS_FROM_PAGE]
        print(f"[INFO] 처리할 상품 링크 {len(links)}개")

        for i, link in enumerate(links, 1):
            print(f"\n[{i}/{len(links)}] {link}")
            rows = crawl_one_product(driver, link)
            rows_for_csv.extend(rows)
    finally:
        driver.quit()

    now = datetime.now().strftime("%Y%m%d_%H%M")
    out = f"kurly_health_merged_{now}.csv"
    cols = [
        "category_url","product_name","brand","product_url",
        "price","review_count","review_date","review_text"
    ]
    pd.DataFrame(rows_for_csv, columns=cols).to_csv(out, index=False, encoding="utf-8-sig")
    print(f"\n[DONE] 통합 CSV 저장 → {out} (행 수: {len(rows_for_csv)})")


if __name__ == "__main__":
    main()


[INFO] 처리할 상품 링크 96개

[1/96] https://www.kurly.com/goods/1001051171

[2/96] https://www.kurly.com/goods/1000963113

[3/96] https://www.kurly.com/goods/5099197

[4/96] https://www.kurly.com/goods/5145668

[5/96] https://www.kurly.com/goods/1000425744

[6/96] https://www.kurly.com/goods/1001325441

[7/96] https://www.kurly.com/goods/1001143403

[8/96] https://www.kurly.com/goods/5055718

[9/96] https://www.kurly.com/goods/5054295

[10/96] https://www.kurly.com/goods/1000127424

[11/96] https://www.kurly.com/goods/1000127407

[12/96] https://www.kurly.com/goods/1000425747

[13/96] https://www.kurly.com/goods/5068121

[14/96] https://www.kurly.com/goods/1001381421

[15/96] https://www.kurly.com/goods/1000425750

[16/96] https://www.kurly.com/goods/5097881

[17/96] https://www.kurly.com/goods/1000217399

[18/96] https://www.kurly.com/goods/1001143409

[19/96] https://www.kurly.com/goods/1000794238

[20/96] https://www.kurly.com/goods/1001051175

[21/96] https://www.kurly.com/goods/100016858

In [14]:
# -*- coding: utf-8 -*-
"""
Kurly 카테고리 1페이지 → 각 상품 상세 이동
=> category_name(소분류: 예 '비타민·미네랄'), product_name, brand, price, review_count, review_date, review_text
=> 한 개 CSV 저장
"""

import json, re, time
from urllib.parse import urlparse
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import undetected_chromedriver as uc
from webdriver_manager.chrome import ChromeDriverManager

# ===== 설정 =====
CATEGORY_URL = "https://www.kurly.com/categories/032009?filters=&page=1"  # 예: 건강식품 > 비타민·미네랄
HEADLESS = False
WINDOW_SIZE = "1920,1080"
SHORT, MID, LONG = 3, 7, 12
SCROLL_STEP = 1200
MAX_PRODUCTS_FROM_PAGE = 10   # 전체는 None
MAX_LOAD_MORE_CLICKS = 25
MAX_REVIEWS_PER_PRODUCT = 300
DATE_PAT = re.compile(r"\d{4}\.\d{1,2}\.\d{1,2}")
DIGIT_ONLY = re.compile(r"^[0-9,]+$")

def make_driver() -> uc.Chrome:
    opts = uc.ChromeOptions()
    if HEADLESS: opts.add_argument("--headless=new")
    opts.add_argument("--disable-dev-shm-usage"); opts.add_argument("--no-sandbox")
    opts.add_argument("--lang=ko_KR"); opts.add_argument(f"--window-size={WINDOW_SIZE}")
    opts.add_argument("--start-maximized"); opts.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    d = uc.Chrome(service=service, options=opts); d.set_page_load_timeout(60); return d

def wait_css(d, css, timeout=MID):  return WebDriverWait(d, timeout).until(EC.presence_of_element_located((By.CSS_SELECTOR, css)))
def wait_xpath(d, xp, timeout=MID): return WebDriverWait(d, timeout).until(EC.presence_of_element_located((By.XPATH, xp)))
def safe_click(d, by, v, timeout=MID):
    try:
        el = WebDriverWait(d, timeout).until(EC.element_to_be_clickable((by, v)))
        d.execute_script("arguments[0].scrollIntoView({block:'center'});", el); time.sleep(0.2); el.click(); return True
    except: return False
def slow_scroll_to_bottom(d, step_px=SCROLL_STEP, sleep_sec=0.6, max_loops=30):
    last = 0
    for _ in range(max_loops):
        d.execute_script(f"window.scrollBy(0,{step_px});"); time.sleep(sleep_sec)
        h = d.execute_script("return document.body.scrollHeight;")
        if h == last: break
        last = h

# ========= (수정) 카테고리 소분류명 추출 =========
def collect_links_and_category_name(d, category_url: str) -> Tuple[List[str], str]:
    d.get(category_url); time.sleep(SHORT)

    # URL 경로(/categories/032009)와 일치하는 a[href*='경로']의 텍스트 = 소분류명
    path = urlparse(category_url).path  # '/categories/032009'
    category_name = None
    try:
        # 상단 소분류 네비게이션에서 같은 경로를 가리키는 a 태그
        els = d.find_elements(By.XPATH, f"//div[@id='container']//a[contains(@href,'{path}')][normalize-space()]")
        # 같은 경로를 가리키는 링크가 여러 개인 경우가 드물게 있어 첫 번째 텍스트 사용
        for a in els:
            t = a.text.strip()
            if t:
                category_name = t
                break
    except:
        category_name = None

    # 보강: 실패 시 og:title/title 등
    if not category_name:
        soup = BeautifulSoup(d.page_source, "lxml")
        tag = soup.select_one('meta[property="og:title"]')
        txt = (tag.get("content") if tag else None) or (soup.title.get_text() if soup.title else "")
        category_name = (re.split(r"[|\-–—]", txt)[0].strip() if txt else "UnknownCategory")

    # 상품 링크 수집
    slow_scroll_to_bottom(d, max_loops=25)
    anchors = d.find_elements(By.CSS_SELECTOR, "a[href*='/goods/']")
    links, seen = [], set()
    for a in anchors:
        href = a.get_attribute("href") or ""
        if "/goods/" in href and href not in seen:
            seen.add(href); links.append(href)
    return links, category_name

# ===== JSON-LD(가격/리뷰수) =====
def parse_json_ld_for_price_reviewcount(html: str) -> Dict[str, Any]:
    soup = BeautifulSoup(html, "lxml"); data = {}
    for tag in soup.select('script[type="application/ld+json"]'):
        try: j = json.loads(tag.string)
        except: continue
        for obj in (j if isinstance(j, list) else [j]):
            if isinstance(obj, dict) and obj.get("@type") == "Product":
                offers = obj.get("offers"); price = None
                if isinstance(offers, dict): price = offers.get("price")
                elif isinstance(offers, list):
                    for off in offers:
                        if isinstance(off, dict) and off.get("price"): price = off["price"]; break
                if price:
                    try: data["price"] = int(str(price).replace(",", "").strip())
                    except: pass
                ar = obj.get("aggregateRating")
                if isinstance(ar, dict) and ar.get("reviewCount") is not None:
                    try: data["review_count"] = int(ar["reviewCount"])
                    except: pass
                return data
    return data

def extract_price_from_dom(d) -> Optional[int]:
    try:
        title = wait_xpath(d, "//h1", timeout=LONG)
        box = title.find_element(By.XPATH, "./ancestor::section[1] | ./ancestor::div[2]")
        spans = box.find_elements(By.XPATH, ".//span[normalize-space()]")
        cand = []
        for s in spans:
            t = s.text.strip()
            if t and "%" not in t and DIGIT_ONLY.match(t): cand.append(int(t.replace(",", "")))
        if cand: return cand[0]
    except: pass
    return None

def extract_reviewcount_from_dom(d) -> Optional[int]:
    try:
        nodes = d.find_elements(By.XPATH, "//*[contains(.,'후기') and (self::a or self::span or self::div)]")
        for n in nodes:
            m = re.search(r"후기[\s(]*(\d+)", (n.text or "").replace(",", ""))
            if m: return int(m.group(1))
    except: pass
    return None

def open_review_tab(d):
    for xp in ["//nav//li[.//span[contains(.,'후기')]]",
               "//nav//*[self::a or self::button][contains(.,'후기')]",
               "//button[contains(.,'후기')]", "//a[contains(.,'후기')]"]:
        if safe_click(d, By.XPATH, xp, timeout=SHORT): time.sleep(1.0); break
    try:
        WebDriverWait(d, LONG).until(EC.presence_of_element_located(
            (By.XPATH, "//*[contains(@aria-label,'상품 후기') or contains(.,'상품 후기')][self::h2 or self::h3 or self::div]")))
    except: pass

def expand_reviews(d, max_clicks=MAX_LOAD_MORE_CLICKS, max_reviews=MAX_REVIEWS_PER_PRODUCT):
    clicks = 0; prev = 0
    while clicks < max_clicks:
        clicked = False
        for b in d.find_elements(By.XPATH, "//button[contains(.,'더보기') or contains(.,'후기 더보기')]"):
            try:
                if b.is_displayed() and b.is_enabled():
                    d.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    time.sleep(0.2); b.click(); clicked = True; clicks += 1; time.sleep(1.2)
            except: pass
        for b in d.find_elements(By.XPATH, "//article//button[contains(.,'더보기') or contains(.,'자세히')]"):
            try:
                if b.is_displayed():
                    d.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    time.sleep(0.15); b.click(); time.sleep(0.2)
            except: pass
        slow_scroll_to_bottom(d, max_loops=2)
        soup = BeautifulSoup(d.page_source, "lxml")
        cur = len(soup.select("article")) + len(soup.select("li"))
        if max_reviews and cur >= max_reviews: break
        if not clicked:
            if cur == prev: break
            prev = cur

def parse_reviews_from_page(d, product_url: str) -> List[Dict[str, Any]]:
    soup = BeautifulSoup(d.page_source, "lxml")
    root = None
    for sel in ["section[id*='review']", "section[class*='review']", "div[id*='review']", "div[class*='review']"]:
        r = soup.select_one(sel); 
        if r: root = r; break
    if root is None: root = soup
    items = root.select("article") or root.select("li")
    rows = []
    for tag in items:
        ps = [p.get_text(" ", strip=True) for p in tag.select("p") if p.get_text(strip=True)]
        body = max(ps, key=len) if ps else tag.get_text(" ", strip=True)
        if not body or len(body) < 10: continue
        m = DATE_PAT.search(tag.get_text(" ", strip=True))
        date_str = m.group(0) if m else None
        rows.append({"product_url": product_url, "review_date": date_str, "review_text": body})
        if MAX_REVIEWS_PER_PRODUCT and len(rows) >= MAX_REVIEWS_PER_PRODUCT: break
    return rows

def split_brand_from_title(title: str) -> Tuple[Optional[str], str]:
    if not title: return None, ""
    m = re.match(r"\s*\[([^\]]+)\]\s*(.*)", title)
    if m:
        brand = m.group(1).strip(); name = m.group(2).strip()
        return brand, (name if name else title.strip())
    return None, title.strip()

def crawl_one_product(d, url: str) -> Tuple[str, str, Optional[int], Optional[int], List[Dict[str, Any]]]:
    d.get(url); time.sleep(SHORT)
    try: title = wait_xpath(d, "//h1", timeout=LONG).text.strip()
    except:
        try: title = wait_css(d, "h1, h2", timeout=MID).text.strip()
        except: title = ""
    brand, product_name = split_brand_from_title(title)
    j = parse_json_ld_for_price_reviewcount(d.page_source)
    price = j.get("price") or extract_price_from_dom(d)
    review_count = j.get("review_count") or extract_reviewcount_from_dom(d)
    open_review_tab(d); expand_reviews(d)
    reviews = parse_reviews_from_page(d, url)
    return product_name, brand, price, review_count, reviews

def main():
    d = make_driver(); rows = []
    try:
        links, category_name = collect_links_and_category_name(d, CATEGORY_URL)
        if isinstance(MAX_PRODUCTS_FROM_PAGE, int): links = links[:MAX_PRODUCTS_FROM_PAGE]
        print(f"[INFO] 소분류 카테고리: {category_name} | 링크 {len(links)}개")
        for i, link in enumerate(links, 1):
            print(f"[{i}/{len(links)}] {link}")
            product_name, brand, price, review_count, reviews = crawl_one_product(d, link)
            if reviews:
                for r in reviews:
                    rows.append({
                        "category_name": category_name,   # 소분류명
                        "product_name": product_name,
                        "brand": brand,
                        "product_url": link,
                        "price": price,
                        "review_count": review_count,
                        "review_date": r.get("review_date"),
                        "review_text": r.get("review_text"),
                    })
            else:
                rows.append({
                    "category_name": category_name,
                    "product_name": product_name,
                    "brand": brand,
                    "product_url": link,
                    "price": price,
                    "review_count": review_count,
                    "review_date": None,
                    "review_text": None,
                })
    finally:
        d.quit()

    out = f"kurly_health_merged_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    cols = ["category_name","product_name","brand","product_url","price","review_count","review_date","review_text"]
    pd.DataFrame(rows, columns=cols).to_csv(out, index=False, encoding="utf-8-sig")
    print(f"[DONE] 저장: {out} (행 {len(rows)})")

if __name__ == "__main__":
    main()


[INFO] 소분류 카테고리: 추천순 | 링크 10개
[1/10] https://www.kurly.com/goods/1001051171
[2/10] https://www.kurly.com/goods/1000963113
[3/10] https://www.kurly.com/goods/5099197
[4/10] https://www.kurly.com/goods/5145668
[5/10] https://www.kurly.com/goods/1000425744
[6/10] https://www.kurly.com/goods/1001325441
[7/10] https://www.kurly.com/goods/1001143403
[8/10] https://www.kurly.com/goods/5055718
[9/10] https://www.kurly.com/goods/5054295
[10/10] https://www.kurly.com/goods/1000127424
[DONE] 저장: kurly_health_merged_20250922_2204.csv (행 99)
